### Preparation

In [ ]:
from google.colab import drive
from collections import Counter
import time
import polars as pl

In [ ]:
drive.mount('/content/drive')

In [4]:
!cp "/content/drive/MyDrive/prog2 - ch4/games_full.parquet" "/content/games_full.parquet"
!cp "/content/drive/MyDrive/prog2 - ch4/moves_full.parquet" "/content/moves_full.parquet"

In [4]:
base_directory_path = '/content/drive/MyDrive/prog2 - chess (Lemu)/chess/'

## Games

In [ ]:
games_file_path = base_directory_path + 'games.csv.gz'
number_of_games_to_load = 10000

In [ ]:
games_dataframe = pd.read_csv(
    games_file_path,
    compression='gzip',
    nrows=number_of_games_to_load
)

In [ ]:
print("games Data:")

print("Games data shape:", games_dataframe.shape)

print(games_dataframe.head())

print("-" * 5)

In [ ]:
print("\nGames Columns:")
print(games_dataframe.columns.tolist())

In [ ]:
print("A games.csv.gz sorainak (partik) összeszámolása folyamatban...\n")

total_games = 0
chunk_counter = 0

chunk_iterator = pd.read_csv(
    games_file_path,
    compression='gzip',
    chunksize=500000,
    usecols=['game_id']
)

for chunk in chunk_iterator:
    chunk_counter += 1
    total_games += len(chunk)

    # Folyamatos visszajelzés menet közben (ezres tagolással)
    print(f"[{chunk_counter}. blokk feldolgozva] Eddig beolvasva: {total_games:,}".replace(',', ' '))

print("\n=======================================")
print(f"KÉSZ! Összes parti a fájlban: {total_games:,}".replace(',', ' '))
print("=======================================")

In [ ]:
import gzip

games_file_path = '/content/drive/MyDrive/prog2 - chess (Lemu)/chess/games.csv.gz'

print("Fájl beolvasása nyers bináris adatként...")

with gzip.open(games_file_path, 'rb') as f:
    # A sum() végigfut a fájlon, és minden sornál hozzáad 1-et
    row_count = sum(1 for _ in f)

# A fejlécet (első sort) nem számoljuk partinak
total_games = row_count - 1

print(f"Összes parti a fájlban: {total_games:,}".replace(',', ' '))

In [ ]:
print("Counting variant types directly from the zipped CSV...\n")
start_time = time.time()

variant_frequency_counter = Counter()
processed_row_count = 0

# Open the file in text mode ('rt') for reading strings
with gzip.open(games_file_path, 'rt', encoding='utf-8') as compressed_file:
    csv_reader = csv.reader(compressed_file)

    # Read the header to dynamically find the index of the 'variant' column
    header_row = next(csv_reader)
    variant_column_index = header_row.index('variant')

    # Iterate through the file row by row
    for row in csv_reader:
        processed_row_count += 1

        # Safety check to ensure the row is not corrupted/too short
        if len(row) > variant_column_index:
            variant_name = row[variant_column_index]
            variant_frequency_counter[variant_name] += 1

        # Give progress feedback every 5 million rows
        if processed_row_count % 5000000 == 0:
            print(f"Processed: {processed_row_count:,} rows...".replace(',', ' '))

end_time = time.time()
total_valid_games = sum(variant_frequency_counter.values())

# Format and print the results
print("\n=======================================")
print("         VARIANT DISTRIBUTION          ")
print("=======================================")
print(f"Total time taken: {end_time - start_time:.1f} seconds\n")

# .most_common() automatically sorts the dictionary by highest values
for variant, count in variant_frequency_counter.most_common():
    percentage = (count / total_valid_games) * 100
    print(f"- {variant:<20} {count:>12,} games ({percentage:>5.2f}%)".replace(',', ' '))

In [ ]:
print(f"Játékváltozatok (variants) eloszlásának vizsgálata {start_date_str} és {end_date_str} között...\n")

# Szótár a változatok számolásához
variant_counts = {}
total_games_in_range = 0

games_chunk_iterator = pd.read_csv(
    games_file_path,
    compression='gzip',
    chunksize=250000,
    usecols=['date', 'variant'] # Csak ez a két oszlop kell, így nagyon gyors lesz
)

for chunk in games_chunk_iterator:
    # Időszakra szűrés (eredménytől függetlenül)
    valid_games = chunk[(chunk['date'] >= start_date_str) & (chunk['date'] <= end_date_str)]

    if len(valid_games) > 0:
        total_games_in_range += len(valid_games)

        # Változatok megszámolása az aktuális chunkban
        chunk_counts = valid_games['variant'].value_counts().to_dict()

        # Hozzáadás az összesített számlálóhoz
        for variant, count in chunk_counts.items():
            variant_counts[variant] = variant_counts.get(variant, 0) + count

    # Korai kilépés (Early Exit), ha elhagytuk a vizsgált időszakot
    if (chunk['date'] < start_date_str).any():
        print("Elértük a céldátum előtti adatokat. Keresés befejezve!\n")
        break

# Eredmények formázása és százalékszámítás
if total_games_in_range > 0:
    print("=======================================")
    print("         VÁLTOZATOK ELOSZLÁSA          ")
    print("=======================================")
    print(f"Összes vizsgált parti: {total_games_in_range}\n")

    # Sorbarendezés gyakoriság szerint (csökkenő)
    sorted_variants = sorted(variant_counts.items(), key=lambda x: x[1], reverse=True)

    for variant, count in sorted_variants:
        percentage = (count / total_games_in_range) * 100
        # Formázott kiíratás: 20 karakter a névnek, 8 a számnak, 2 tizedesjegy a százaléknak
        print(f"- {variant:<20} {count:>8} parti ({percentage:>5.2f}%)")
else:
    print("Nem találtunk partit a megadott időszakban.")

In [ ]:
# Define the file path for the games dataset
games_file_path = '/content/drive/MyDrive/prog2 - chess (Lemu)/chess/games.csv.gz'

# Use a set to automatically store only unique dates
all_unique_dates = set()

# Read the file in chunks, but ONLY load the 'date' column to save memory and time
chunk_iterator = pd.read_csv(
    games_file_path,
    compression='gzip',
    chunksize=250000,
    usecols=['date']
)

for current_chunk in chunk_iterator:
    # Get unique dates from the current chunk
    chunk_unique_dates = current_chunk['date'].unique()

    # Add them to the main set (duplicates are automatically ignored)
    all_unique_dates.update(chunk_unique_dates)

# Convert the set to a sorted list to view them chronologically
sorted_unique_dates = sorted(list(all_unique_dates))

print(f"Total number of unique dates: {len(sorted_unique_dates)}")
print("First 10 dates:", sorted_unique_dates[:10])
print("Last 10 dates:", sorted_unique_dates[-10:])

## Tournaments

In [ ]:
tournaments_file_path = base_directory_path + 'tournaments.csv.gz'
tournaments_dataframe = pd.read_csv(tournaments_file_path, compression='gzip')

In [ ]:
print("Tournaments Data:")

print("Tournaments data shape:", tournaments_dataframe.shape)

print(tournaments_dataframe.head())

print("-" * 50)

In [ ]:
print("Tournaments Columns:")
print(tournaments_dataframe.columns.tolist())

In [ ]:
# Convert the UNIX timestamps (in milliseconds) to readable datetime objects
tournaments_dataframe['start_datetime'] = pd.to_datetime(tournaments_dataframe['startsAt'], unit='ms')
tournaments_dataframe['finish_datetime'] = pd.to_datetime(tournaments_dataframe['finishesAt'], unit='ms')

# Now you can see the exact date and time the tournament happened
print(tournaments_dataframe[['id', 'fullName', 'start_datetime', 'finish_datetime']].head())

## Moves

In [ ]:
moves_file_path = base_directory_path + 'moves.csv.gz'
number_of_moves_to_load = 5000

In [ ]:
partial_moves_dataframe = pd.read_csv(
    moves_file_path,
    compression='gzip',
    nrows=number_of_moves_to_load
)

In [ ]:
print("Partial Moves Data:")

print("Partial moves data shape:", partial_moves_dataframe.shape)

print(partial_moves_dataframe.head())

In [ ]:
# counting the number of observations (rows)

massive_moves_file_path = '/content/drive/MyDrive/prog2 - chess (Lemu)/chess/moves.csv.gz'

rows_per_chunk = 500000
total_row_count = 0

dataframe_chunk_iterator = pd.read_csv(
    massive_moves_file_path,
    compression='gzip',
    chunksize=rows_per_chunk
)

for data_chunk in dataframe_chunk_iterator:
    # Map step: calculate length of current chunk
    current_chunk_length = len(data_chunk)

    # Reduce step: accumulate the total
    total_row_count += current_chunk_length

    print(f"Processed a chunk. Current total: {total_row_count}")

print(f"Final total number of rows: {total_row_count}")

## 2026-04-13 -- number of tournaments, games and moves this day

In [ ]:
# Define the base directory and file paths
base_directory_path = '/content/drive/MyDrive/prog2 - chess (Lemu)/chess/'
tournaments_file_path = base_directory_path + 'tournaments.csv.gz'
games_file_path = base_directory_path + 'games.csv.gz'
moves_file_path = base_directory_path + 'moves.csv.gz'

# Define target dates
target_date_iso = '2026-04-13'
target_date_str = '2026.04.13'

# --- 1. Calculate number of tournaments ---
print("Scanning tournaments...")
tournaments_dataframe = pd.read_csv(tournaments_file_path, compression='gzip')
tournaments_dataframe['start_datetime'] = pd.to_datetime(tournaments_dataframe['startsAt'], unit='ms')

# Filter for the target date
tournaments_on_target = tournaments_dataframe[tournaments_dataframe['start_datetime'].dt.date == pd.to_datetime(target_date_iso).date()]
number_of_tournaments = len(tournaments_on_target)


# --- 2. Calculate number of games (Early Exit via Date String) ---
print("Scanning games using Early Exit...")
number_of_games = 0
target_game_ids = set()

games_chunk_iterator = pd.read_csv(
    games_file_path,
    compression='gzip',
    chunksize=250000,
    usecols=['game_id', 'date']
)

for games_chunk in games_chunk_iterator:
    # EARLY EXIT LOGIC: If we see a date older than our target date
    if (games_chunk['date'] < target_date_str).any():

        # Grab any remaining games from our target date inside this specific chunk
        valid_games = games_chunk[games_chunk['date'] == target_date_str]
        number_of_games += len(valid_games)
        target_game_ids.update(valid_games['game_id'].tolist())

        print(f"Reached data older than {target_date_str}. Breaking games loop early!")
        break

    # Standard check: If we haven't hit the early exit yet, look for our target date
    valid_games = games_chunk[games_chunk['date'] == target_date_str]
    if len(valid_games) > 0:
        number_of_games += len(valid_games)
        target_game_ids.update(valid_games['game_id'].tolist())


# --- 3. Calculate number of moves (Early Exit via Empty Chunk Counter) ---
print("Scanning moves using Early Exit...")
number_of_moves = 0
found_target_data = False
empty_chunk_counter = 0

moves_chunk_iterator = pd.read_csv(
    moves_file_path,
    compression='gzip',
    chunksize=500000,
    usecols=['game_id']
)

for moves_chunk in moves_chunk_iterator:
    # Find overlap using the set of game IDs we just extracted
    overlap_moves = moves_chunk[moves_chunk['game_id'].isin(target_game_ids)]

    if len(overlap_moves) > 0:
        found_target_data = True
        empty_chunk_counter = 0 # Reset counter if we find data
        number_of_moves += len(overlap_moves)

    # EARLY EXIT LOGIC: If we previously found our data, but now we are seeing empty chunks
    elif found_target_data and len(overlap_moves) == 0:
        empty_chunk_counter += 1

        # If we hit 3 completely empty chunks in a row, we assume we have passed the date block
        if empty_chunk_counter > 3:
            print("Passed target date block in moves file. Breaking early!")
            break

# --- Final Results ---
print("\n--- Final Statistics for 2026-04-13 ---")
print(f"Number of tournaments: {number_of_tournaments}")
print(f"Number of games: {number_of_games}")
print(f"Number of moves: {number_of_moves}")

## number of games ended with 3-fold-repetition between 2024-02-04 and 2024-02-10

In [ ]:
# Define file paths
base_directory_path = '/content/drive/MyDrive/prog2 - chess (Lemu)/chess/'
games_file_path = base_directory_path + 'games.csv.gz'
moves_file_path = base_directory_path + 'moves.csv.gz'

# Define our target date
start_date_str = '2024-06-04'
end_date_str = '2024-06-10'

In [ ]:
# =====================================================================
# --- STEP 1 & 2: Extract and Merge Data using DuckDB
# =====================================================================
# This single SQL query replaces all previous chunking logic.
# It automatically joins the files, filters the dates and results,
# and orders the moves chronologically for each game.
extraction_query = f"""
    SELECT
        moves_table.game_id,
        moves_table.move,
        games_table.site
    FROM read_csv_auto('{moves_file_path}') AS moves_table
    JOIN read_csv_auto('{games_file_path}') AS games_table
      ON moves_table.game_id = games_table.game_id
    WHERE games_table.date >= '{start_date_str}'
      AND games_table.date <= '{end_date_str}'
      AND games_table.result = '1/2-1/2'
    ORDER BY moves_table.game_id, moves_table.move_no
"""

# Execute the query and load the filtered results into a Pandas DataFrame
filtered_moves_dataframe = duckdb.query(extraction_query).df()
unique_game_count = filtered_moves_dataframe['game_id'].nunique()

print(f"Extraction complete. Found {unique_game_count} drawn games.")

In [ ]:
# =====================================================================
# --- STEP 3: State Tracking with Python-Chess
# =====================================================================
print("\nSimulating board states to find actual 3-fold repetitions...")

threefold_repetition_count = 0
successful_simulations = 0
crashed_simulations = 0
repetition_urls = []

# Group the DataFrame by game_id and site so we can process one game at a time
grouped_games = filtered_moves_dataframe.groupby(['game_id', 'site'])

for (current_game_id, current_site_url), game_moves_data in grouped_games:
    board_tracker = chess.Board()
    game_crashed = False
    has_actual_repetition = False

    # Extract the moves into a sequential list
    move_list = game_moves_data['move'].tolist()

    # Play out the game
    for move_str in move_list:
        try:
            board_tracker.push_san(move_str)

            # Trust only physical, actual occurrences
            if board_tracker.is_repetition(3):
                has_actual_repetition = True

        except ValueError:
            game_crashed = True
            break

    # Tally the simulation results
    if game_crashed:
        crashed_simulations += 1
    else:
        successful_simulations += 1

        if has_actual_repetition:
            threefold_repetition_count += 1
            repetition_urls.append(current_site_url)

# Print Final Statistics
print("\n=======================================")
print("             FINAL RESULT              ")
print("=======================================")
print(f"Timeframe: {start_date_str} to {end_date_str}")
print(f"Successfully simulated games: {successful_simulations}")
print(f"Crashed games (Bad Data / Fragments): {crashed_simulations}")
print(f"True 3-fold repetitions found: {threefold_repetition_count}")

print("\n--- MANUAL VERIFICATION (First 15) ---")
for url in repetition_urls[:15]:
    print(f"- {url}")

## csv.gz to parquet

In [ ]:
import duckdb
import time

# Define the source CSV path (read-only)
csv_path = '/content/drive/MyDrive/prog2 - chess (Lemu)/chess/moves.csv.gz'

# Define the target Parquet path in your new folder
parquet_path = '/content/drive/MyDrive/prog2 - ch4/moves_full.parquet'

In [ ]:
print("🚀 Starting full database conversion to Parquet...")
print(f"Target file: {parquet_path}")
print("WARNING: This process will read ~4 billion rows. It may take 1 to 1.5 hours.")
print("Please keep this browser tab open and active to prevent Colab from disconnecting.\n")

start_time = time.time()

# Execute the conversion for the entire file (no LIMIT applied)
duckdb.query(f"COPY (SELECT * FROM read_csv_auto('{csv_path}')) TO '{parquet_path}' (FORMAT PARQUET)")

end_time = time.time()
total_minutes = (end_time - start_time) / 60

print("==================================================")
print("✅ SUCCESS! The massive conversion is complete.")
print(f"Total time taken: {total_minutes:.1f} minutes")
print("You now have a highly optimized database ready for instant queries!")
print("==================================================")

In [ ]:
# Define source paths (read-only shared folder)
games_csv = '/content/drive/MyDrive/prog2 - chess (Lemu)/chess/games.csv.gz'
tournaments_csv = '/content/drive/MyDrive/prog2 - chess (Lemu)/chess/tournaments.csv.gz'

# Define target paths in your personal folder
games_parquet = '/content/drive/MyDrive/prog2 - ch4/games_full.parquet'
tournaments_parquet = '/content/drive/MyDrive/prog2 - ch4/tournaments_full.parquet'

def convert_to_parquet(source, target, description):
    print(f"Starting conversion: {description}...")
    start_time = time.time()

    # Use DuckDB to convert CSV to Parquet
    duckdb.query(f"COPY (SELECT * FROM read_csv_auto('{source}')) TO '{target}' (FORMAT PARQUET)")

    duration = time.time() - start_time
    print(f"Finished! Saved to: {target}")
    print(f"Time taken: {duration:.2f} seconds\n")

# Run the conversions
convert_to_parquet(games_csv, games_parquet, "Games Database")
convert_to_parquet(tournaments_csv, tournaments_parquet, "Tournaments Database")

print("==================================================")
print("All auxiliary files are now optimized in Parquet!")
print("==================================================")

## Question 8
#### Hány alkalommal végződött döntetlennel a játszma március 20-án olyan esetekben, ahol a mérkőzést lezáró utolsó lépés egy gyalog vezérré történő átváltozása volt?

In [ ]:
import duckdb
import time

moves_path = '/content/drive/MyDrive/prog2 - ch4/moves_full.parquet'
games_path = '/content/drive/MyDrive/prog2 - ch4/games_full.parquet'

In [ ]:
print("🚀 Sakk-szabály alapú, golyóálló szűrés indítása...")
start_time = time.time()

# 1. Március 20-i döntetlenek
target_games_df = duckdb.query(f"""
    SELECT game_id, site, date
    FROM '{games_path}'
    WHERE CAST(date AS VARCHAR) LIKE '%-03-20'
      AND result = '1/2-1/2'
""").df()

# 2. A zseniális sakk-logika: Egyetlen szöveggé fűzzük a teljes lépést (pl. "a8=Q Ke6")
query = f"""
    WITH MoveCounts AS (
        SELECT
            m.game_id,
            m.move_no,
            COUNT(*) as moves_in_this_turn,
            -- Összefűzzük a világos és sötét lépését
            STRING_AGG(m.move, ' ') as all_moves_in_turn
        FROM '{moves_path}' m
        JOIN target_games_df t ON m.game_id = t.game_id
        GROUP BY m.game_id, m.move_no
    ),
    MaxMoveNo AS (
        SELECT game_id, MAX(move_no) as final_move_no
        FROM MoveCounts
        GROUP BY game_id
    ),
    TerminalTurns AS (
        SELECT
            mc.game_id,
            mc.moves_in_this_turn,
            mc.all_moves_in_turn
        FROM MoveCounts mc
        JOIN MaxMoveNo mm ON mc.game_id = mm.game_id AND mc.move_no = mm.final_move_no
    )
    SELECT
        tt.game_id,
        tt.all_moves_in_turn as last_turn,
        CASE
            WHEN tt.moves_in_this_turn = 1 THEN 'White'
            ELSE 'Black'
        END as finishing_side
    FROM TerminalTurns tt
    WHERE
        -- Szabály 1: Csak Világos lépett az utolsó körben, ÉS ez egy 8. sori átváltozás
        (tt.moves_in_this_turn = 1 AND tt.all_moves_in_turn LIKE '%8=Q%')
        OR
        -- Szabály 2: Mindketten léptek (tehát Sötét volt az utolsó), ÉS volt 1. sori átváltozás
        (tt.moves_in_this_turn = 2 AND tt.all_moves_in_turn LIKE '%1=Q%')
"""

final_moves_df = duckdb.query(query).df()

# 3. Végeredmény összefésülése
final_results = final_moves_df.merge(target_games_df, on='game_id')

duration = time.time() - start_time
print(f"🏁 Elemzés befejezve: {duration:.2f} másodperc.")
print(f"VALÓDI, végső átváltozások száma: {len(final_results)}\n")

if not final_results.empty:
    white_wins = len(final_results[final_results['finishing_side'] == 'White'])
    black_wins = len(final_results[final_results['finishing_side'] == 'Black'])

    print(f"Megoszlás - Világos: {white_wins}, Sötét: {black_wins}\n")
    print(final_results[['date', 'finishing_side', 'last_turn', 'site']].sort_values('date', ascending=False).to_string(index=False))
else:
    print("Nulla találat. Ezzel a 100%-ig pontos sakk-logikával kiderült, hogy nincs is ilyen parti.")

In [ ]:
import duckdb

# File paths
moves_parquet = '/content/drive/MyDrive/prog2 - ch4/moves_full.parquet'
games_parquet = '/content/drive/MyDrive/prog2 - ch4/games_full.parquet'

# These are the site IDs from the URLs
site_codes = ('Ev3X6DXJ', 'ChxhaUD5', 'iWZZq7Zo')

print(f"🕵️ Scanning for site IDs: {site_codes} in the 'site' column...\n")

# 1. Step: Find the internal game_ids and basic info
query_find_ids = f"""
    SELECT game_id, site, result, termination
    FROM '{games_parquet}'
    WHERE site LIKE '%Ev3X6DXJ%'
       OR site LIKE '%ChxhaUD5%'
       OR site LIKE '%iWZZq7Zo%'
"""
target_games = duckdb.query(query_find_ids).df()

if target_games.empty:
    print("❌ Still nothing found. Check if the file paths or IDs are correct!")
else:
    print("--- GAME METADATA FOUND ---")
    print(target_games.to_string(index=False))

    # Get the internal IDs for the move lookup
    internal_ids = tuple(target_games['game_id'].tolist())

    print("\n--- ALL RECORDED MOVES (Source of Truth) ---")
    # 2. Step: Show the last 5 plies for these games
    query_moves = f"""
        SELECT game_id, move_no, move
        FROM '{moves_parquet}'
        WHERE game_id IN {internal_ids}
        QUALIFY ROW_NUMBER() OVER (PARTITION BY game_id ORDER BY move_no DESC) <= 5
        ORDER BY game_id, move_no ASC
    """
    print(duckdb.query(query_moves).df().to_string(index=False))

## Question 6
### Hány játszma végződött háromszori lépésismétlés miatt döntetlennel 2024.03.12. és 2024.11.19. között csak „standard” alapállású partikat figyelembe véve?

In [ ]:
import duckdb

# A játszmákat tartalmazó fájl elérési útja
games_parquet_utvonal = '/content/drive/MyDrive/prog2 - ch4/games_full.parquet'

print("🔍 Adatbázis mezőinek feltérképezése...\n")

# 1. A létező játéktípusok (variant) lekérdezése
lekerdezes_jatektipusok = f"""
    SELECT DISTINCT variant
    FROM '{games_parquet_utvonal}'
    LIMIT 20
"""
egyedi_jatektipusok_df = duckdb.query(lekerdezes_jatektipusok).df()
print("--- LÉTEZŐ JÁTÉKTÍPUSOK (variant) ---")
print(egyedi_jatektipusok_df.to_string(index=False))

print("\n")

# 2. A döntetlennel végződő játszmák befejezési okainak (termination) lekérdezése
lekerdezes_dontetlen_okok = f"""
    SELECT DISTINCT termination
    FROM '{games_parquet_utvonal}'
    WHERE result = '1/2-1/2'
"""
egyedi_dontetlen_okok_df = duckdb.query(lekerdezes_dontetlen_okok).df()
print("--- DÖNTETLEN OKOK (termination) ---")
print(egyedi_dontetlen_okok_df.to_string(index=False))

## Question 14
### Mely napokon (datumokon) történt legalább egy olyan játszma, ahol az eredetileg az a2 mezőn álló világos gyalog eljutott a g8 mezőre és ott átváltozott? (elso 10 datum)

## Question 9
### Melyik felhasználó(k) szenvedte(k) el a legtöbb vereséget időtúllépés miatt olyan játszmákban, ahol a mérkőzés kezdetén élt(ek) a 'Berserk' opcióval? (holtverseny esetén abc szerint első 10)

In [ ]:
import duckdb

# File path
games_parquet = '/content/drive/MyDrive/prog2 - ch4/games_full.parquet'

print("🔍 Exploring table schema and player metadata...\n")

# 1. Get all column names to see if 'player' or 'berserk' columns exist
column_names = duckdb.query(f"DESCRIBE SELECT * FROM '{games_parquet}'").df()
print("--- ALL AVAILABLE COLUMNS ---")
print(column_names['column_name'].tolist())

print("\n--- SAMPLE DATA (Top 5 rows) ---")
# 2. Peek into the data to see the content of these columns
# We select all columns to be sure we don't miss anything
sample_data = duckdb.query(f"SELECT * FROM '{games_parquet}' LIMIT 5").df()
print(sample_data.to_string())

print("\n--- CHECKING FOR 'TIME FORFEIT' AND POTENTIAL BERSERK VALUES ---")
# 3. Check unique values in termination to confirm 'Time forfeit'
# and see if there are any columns containing 'berserk' in their name
berserk_cols = [col for col in sample_data.columns if 'berserk' in col.lower()]
if berserk_cols:
    print(f"✅ Found potential Berserk columns: {berserk_cols}")
else:
    print("⚠️ No columns with 'berserk' in their name were found.")

In [ ]:
import duckdb

games_parquet_path = '/content/drive/MyDrive/prog2 - ch4/games_full.parquet'

In [ ]:
print("🚀 Calculating Time Forfeit losses for Berserk players...\n")

query = f"""
    WITH BerserkLosses AS (
        -- Check White players who lost on time and berserked
        SELECT white AS player
        FROM '{games_parquet_path}'
        WHERE termination = 'Time forfeit'
          AND result = '0-1' -- White lost
          AND timecontrol LIKE '%+%' -- Ensure valid time control format
          -- Convert 'whitestart' HH:MM:SS format to total seconds and compare to base time
          AND (
              EXTRACT(HOUR FROM CAST(whitestart AS TIME)) * 3600 +
              EXTRACT(MINUTE FROM CAST(whitestart AS TIME)) * 60 +
              EXTRACT(SECOND FROM CAST(whitestart AS TIME))
          ) < CAST(SPLIT_PART(timecontrol, '+', 1) AS INTEGER)

        UNION ALL

        -- Check Black players who lost on time and berserked
        SELECT black AS player
        FROM '{games_parquet_path}'
        WHERE termination = 'Time forfeit'
          AND result = '1-0' -- Black lost
          AND timecontrol LIKE '%+%'
          -- Convert 'blackstart' HH:MM:SS format to total seconds and compare to base time
          AND (
              EXTRACT(HOUR FROM CAST(blackstart AS TIME)) * 3600 +
              EXTRACT(MINUTE FROM CAST(blackstart AS TIME)) * 60 +
              EXTRACT(SECOND FROM CAST(blackstart AS TIME))
          ) < CAST(SPLIT_PART(timecontrol, '+', 1) AS INTEGER)
    )

    -- Aggregate the results, sort by losses descending, then alphabetically, top 10
    SELECT
        player,
        COUNT(*) AS loss_count
    FROM BerserkLosses
    GROUP BY player
    ORDER BY loss_count DESC, player ASC
    LIMIT 10
"""

# Execute and display the exact Top 10 list
result_df = duckdb.query(query).df()
print("--- TOP 10 PLAYERS (Berserk Time Forfeit Losses) ---")
print(result_df.to_string(index=False))

In [ ]:
import duckdb

games_parquet_path = '/content/drive/MyDrive/prog2 - ch4/games_full.parquet'

# A top 10 játékos listája
top_players = (
    'zelkovahi', 'Berserk13cako', 'ramiz-dema', 'Talant74',
    'vidyasagar2017', 'Noahchi123', 'account4BD', 'GoingDownSouth',
    'koryusuf', 'ZerkingQueen'
)

print("🔍 1. Rész: Minta partik (Berserk + Időtúllépés) letöltése ellenőrzéshez...\n")

url_query = f"""
    WITH BerserkLosses AS (
        SELECT white AS player, site, timecontrol, whitestart as start_time
        FROM '{games_parquet_path}'
        WHERE white IN {top_players}
          AND termination = 'Time forfeit' AND result = '0-1'
          AND timecontrol LIKE '%+%'
          AND (EXTRACT(HOUR FROM CAST(whitestart AS TIME)) * 3600 +
               EXTRACT(MINUTE FROM CAST(whitestart AS TIME)) * 60 +
               EXTRACT(SECOND FROM CAST(whitestart AS TIME))) < CAST(SPLIT_PART(timecontrol, '+', 1) AS INTEGER)

        UNION ALL

        SELECT black AS player, site, timecontrol, blackstart as start_time
        FROM '{games_parquet_path}'
        WHERE black IN {top_players}
          AND termination = 'Time forfeit' AND result = '1-0'
          AND timecontrol LIKE '%+%'
          AND (EXTRACT(HOUR FROM CAST(blackstart AS TIME)) * 3600 +
               EXTRACT(MINUTE FROM CAST(blackstart AS TIME)) * 60 +
               EXTRACT(SECOND FROM CAST(blackstart AS TIME))) < CAST(SPLIT_PART(timecontrol, '+', 1) AS INTEGER)
    )
    SELECT player, site, timecontrol, start_time
    FROM BerserkLosses
    LIMIT 10
"""

urls_df = duckdb.query(url_query).df()
print(urls_df.to_string(index=False))

print("\n--------------------------------------------------\n")
print("📊 2. Rész: Összes játszma számolása ezeknél a játékosoknál...\n")

total_games_query = f"""
    SELECT
        player,
        COUNT(*) AS total_games_played
    FROM (
        SELECT white AS player FROM '{games_parquet_path}' WHERE white IN {top_players}
        UNION ALL
        SELECT black AS player FROM '{games_parquet_path}' WHERE black IN {top_players}
    )
    GROUP BY player
    ORDER BY total_games_played DESC
"""

totals_df = duckdb.query(total_games_query).df()
print(totals_df.to_string(index=False))

## Question 16
### leghosszabb döntetlen széria

In [ ]:
import duckdb
import time

games_parquet_path = '/content/drive/MyDrive/prog2 - ch4/games_full.parquet'

In [ ]:
print("🚀 Starting 'Gaps and Islands' analysis for the longest draw streak...")
start_time = time.time()

query = f"""
WITH PlayerGames AS (
    -- Step 1: Normalize data so each player gets one row per game played
    -- We extract Elo as integer, handling '?' provisional ratings safely
    SELECT
        white AS player,
        TRY_CAST(whiteelo AS INTEGER) AS elo,
        utcdate,
        utctime,
        CASE WHEN result = '1/2-1/2' THEN 1 ELSE 0 END AS is_draw
    FROM '{games_parquet_path}'
    WHERE variant = 'Standard'

    UNION ALL

    SELECT
        black AS player,
        TRY_CAST(blackelo AS INTEGER) AS elo,
        utcdate,
        utctime,
        CASE WHEN result = '1/2-1/2' THEN 1 ELSE 0 END AS is_draw
    FROM '{games_parquet_path}'
    WHERE variant = 'Standard'
),
ChronologicalGames AS (
    -- Step 2: Assign row numbers ordered by strict time to find consecutive sequences
    SELECT
        player,
        elo,
        utcdate,
        utctime,
        is_draw,
        -- Absolute chronological row number for all games of this player
        ROW_NUMBER() OVER (PARTITION BY player ORDER BY utcdate ASC, utctime ASC) as rn_total,
        -- Row number counting ONLY the draws of this player
        ROW_NUMBER() OVER (PARTITION BY player, is_draw ORDER BY utcdate ASC, utctime ASC) as rn_draws
    FROM PlayerGames
),
StreakGroups AS (
    -- Step 3: The 'Gaps and Islands' subtraction trick
    -- Consecutive draws will end up having the exact same 'streak_id'
    SELECT
        player,
        elo,
        utcdate,
        utctime,
        (rn_total - rn_draws) as streak_id
    FROM ChronologicalGames
    WHERE is_draw = 1
),
AggregatedStreaks AS (
    -- Step 4: Calculate length, dates, and final Elo for each isolated streak
    SELECT
        player,
        COUNT(*) as streak_length,
        MIN(utcdate) as start_date,
        MAX(utcdate) as end_date,
        -- Fetch the exact Elo rating from the game that has the latest timestamp in this streak
        arg_max(elo, utcdate || ' ' || utctime) as end_elo
    FROM StreakGroups
    GROUP BY player, streak_id
)
-- Step 5: Output the absolute winner applying the tie-breaker
SELECT
    player,
    start_date,
    end_date,
    streak_length,
    end_elo
FROM AggregatedStreaks
ORDER BY streak_length DESC, end_elo DESC NULLS LAST
LIMIT 1
"""

result_df = duckdb.query(query).df()
duration = time.time() - start_time

print(f"🏁 Analysis finished in {duration:.2f} seconds.")
print("\n--- THE LONGEST DRAW STREAK WINNER ---")
print(result_df.to_string(index=False))

## Q18
### leghosszabb nyeretlenségi széria

In [ ]:
import duckdb
import time

games_parquet_path = '/content/drive/MyDrive/prog2 - ch4/games_full.parquet'

In [ ]:
print("🚀 Starting 'Gaps and Islands' analysis for the longest winless streak...")
start_time = time.time()

query = f"""
WITH PlayerGames AS (
    -- Step 1: Normalize data. Winless means the player got a draw or lost the game.
    SELECT
        white AS player,
        utcdate,
        utctime,
        CASE WHEN result IN ('1/2-1/2', '0-1') THEN 1 ELSE 0 END AS is_winless
    FROM '{games_parquet_path}'
    WHERE variant = 'Standard'

    UNION ALL

    SELECT
        black AS player,
        utcdate,
        utctime,
        CASE WHEN result IN ('1/2-1/2', '1-0') THEN 1 ELSE 0 END AS is_winless
    FROM '{games_parquet_path}'
    WHERE variant = 'Standard'
),
ChronologicalGames AS (
    -- Step 2: Assign row numbers ordered by strict time to find consecutive sequences
    SELECT
        player,
        utcdate,
        utctime,
        is_winless,
        ROW_NUMBER() OVER (PARTITION BY player ORDER BY utcdate ASC, utctime ASC) as rn_total,
        ROW_NUMBER() OVER (PARTITION BY player, is_winless ORDER BY utcdate ASC, utctime ASC) as rn_winless
    FROM PlayerGames
),
StreakGroups AS (
    -- Step 3: The 'Gaps and Islands' subtraction trick for winless games
    SELECT
        player,
        utcdate,
        utctime,
        (rn_total - rn_winless) as streak_id
    FROM ChronologicalGames
    WHERE is_winless = 1
),
AggregatedStreaks AS (
    -- Step 4: Calculate length and dates for each isolated streak
    SELECT
        player,
        COUNT(*) as streak_length,
        MIN(utcdate) as start_date,
        MAX(utcdate) as end_date
    FROM StreakGroups
    GROUP BY player, streak_id
)
-- Step 5: Output the absolute winner applying the unique tie-breaker
-- The tie-breaker prioritizes names strictly coming after 'lili' alphabetically
SELECT
    player,
    start_date,
    end_date,
    streak_length
FROM AggregatedStreaks
ORDER BY
    streak_length DESC,
    CASE
        WHEN LOWER(player) > 'lili' THEN LOWER(player)
        ELSE 'zzzzzzzz' -- Push names coming before 'Lili' to the bottom
    END ASC
LIMIT 1
"""

result_df = duckdb.query(query).df()
duration = time.time() - start_time

print(f"🏁 Analysis finished in {duration:.2f} seconds.")
print("\n--- THE LONGEST WINLESS STREAK WINNER ---")
print(result_df.to_string(index=False))

## Q13

In [11]:
import polars as pl
import time

# ==========================================
# ⚡ THE NEW FAST LOCAL PATHS ⚡
# ==========================================
games_parquet_path = '/content/games_full.parquet'
moves_parquet_path = '/content/moves_full.parquet'

In [ ]:
print("🚀 Starting Polars engine on LOCAL NVMe SSD...")
start_time_exec = time.time()

# ==========================================
# PHASE 1: Independent Linear Scans
# ==========================================
print("⏳ Phase 1: Scanning 0.5% of Games & Moves...")
t0 = time.time()

# We can safely use 0.5% (divisor 200) because local RAM and SSD can handle it easily
games_df = pl.scan_parquet(games_parquet_path).filter(
    (pl.col('result').is_in(['1-0', '0-1'])) &
    ((pl.col('game_id').hash() % 200) == 0)
).select([
    'game_id', 'result',
    pl.col('whitestart').cast(pl.String).alias('nominal_white_start'),
    pl.col('blackstart').cast(pl.String).alias('nominal_black_start')
]).collect()

moves_df = pl.scan_parquet(moves_parquet_path).filter(
    ((pl.col('game_id').hash() % 200) == 0)
).select([
    'game_id', 'move_no',
    pl.col('clock').cast(pl.String).alias('clock_str')
]).collect(streaming=True)

print(f"   [Data extracted from SSD in {time.time() - t0:.2f} seconds]")


In [ ]:
# ==========================================
# PHASE 2: In-Memory Aggregation
# ==========================================
print("⏳ Phase 2: Joining and aggregating in RAM...")
t2 = time.time()

joined_df = moves_df.join(games_df, on='game_id', how='inner')

game_summary_df = joined_df.group_by(
    ['game_id', 'move_no', 'result', 'nominal_white_start', 'nominal_black_start']
).agg(
    pl.col('clock_str').alias('clock_list')
).sort(['game_id', 'move_no']).group_by('game_id').agg([
    pl.col('result').first().alias('result'),
    pl.col('nominal_white_start').first().alias('nominal_white_start'),
    pl.col('nominal_black_start').first().alias('nominal_black_start'),

    pl.col('clock_list').first().list.first().alias('first_white_tick'),
    pl.col('clock_list').last().list.first().alias('final_white_tick'),

    pl.col('clock_list').filter(pl.col('clock_list').list.len() > 1).first().list.get(1).alias('first_black_tick'),
    pl.col('clock_list').filter(pl.col('clock_list').list.len() > 1).last().list.get(1).alias('final_black_tick')
])

print(f"   [Aggregation completed in {time.time() - t2:.2f} seconds]")

In [ ]:
# ==========================================
# PHASE 3: Python Logic (Berserk Deduction)
# ==========================================
print("⏳ Phase 3: Running final Berserk and time math...")
t3 = time.time()

def parse_clock_to_sec(clock_val):
    if clock_val is None: return 0.0
    parts = str(clock_val).split(':')
    if len(parts) == 3:
        return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
    return 0.0

more_time_used_wins = 0
less_time_used_wins = 0
valid_games = 0

for row in game_summary_df.iter_rows(named=True):
    match_result = row['result']

    nominal_white = parse_clock_to_sec(row['nominal_white_start'])
    nominal_black = parse_clock_to_sec(row['nominal_black_start'])

    first_w_str = row['first_white_tick']
    final_w_str = row['final_white_tick']

    if first_w_str is None:
        white_time_spent = 0.0
    else:
        first_w = parse_clock_to_sec(first_w_str)
        final_w = parse_clock_to_sec(final_w_str)
        true_w_start = nominal_white / 2 if first_w <= (nominal_white / 2) else nominal_white
        white_time_spent = true_w_start - final_w

    first_b_str = row['first_black_tick']
    final_b_str = row['final_black_tick']

    if first_b_str is None:
        black_time_spent = 0.0
    else:
        first_b = parse_clock_to_sec(first_b_str)
        final_b = parse_clock_to_sec(final_b_str)
        true_b_start = nominal_black / 2 if first_b <= (nominal_black / 2) else nominal_black
        black_time_spent = true_b_start - final_b

    if white_time_spent <= 0 and black_time_spent <= 0:
        continue
    if white_time_spent == black_time_spent:
        continue

    valid_games += 1
    white_used_more_time = white_time_spent > black_time_spent

    if (match_result == '1-0' and white_used_more_time) or (match_result == '0-1' and not white_used_more_time):
        more_time_used_wins += 1
    else:
        less_time_used_wins += 1

print(f"   [Math logic completed in {time.time() - t3:.2f} seconds]")

# Final Output
execution_time = time.time() - start_time_exec
print(f"\n🏁 Total Analysis finished in {execution_time:.2f} seconds.")

if valid_games > 0:
    more_pct = (more_time_used_wins / valid_games) * 100
    less_pct = (less_time_used_wins / valid_games) * 100
    print("\n📊 FINAL STATISTICS (Based on Polars 0.5% sample):")
    print(f"Total valid decisive games analyzed: {valid_games}")
    print("-" * 60)
    print(f"Winners who used MORE time: {more_time_used_wins} games ({more_pct:.2f}%)")
    print(f"Winners who used LESS time: {less_time_used_wins} games ({less_pct:.2f}%)")
else:
    print("No valid decisive games found in the sample.")

### Q20

In [6]:
import polars as pl
import time

games_parquet_path = '/content/games_full.parquet'
moves_parquet_path = '/content/moves_full.parquet'

In [ ]:
print("🚀 Starting Yearly Queen's Gambit Percentage Analysis (April 21 - May 18)...")
start_time_total = time.time()

# ==========================================
# PHASE 1: Identify and Count ALL Standard Spring Games per Year
# ==========================================
print("⏳ Phase 1: Analyzing Spring games distribution by year...")

# Process the games table to find all relevant IDs and their corresponding years
spring_games_metadata = pl.scan_parquet(games_parquet_path).filter(
    pl.col('variant') == 'Standard'
).with_columns([
    pl.col('utcdate').dt.combine(pl.col('utctime')).alias('utc_datetime')
]).with_columns([
    pl.col('utc_datetime').dt.replace_time_zone('UTC').dt.convert_time_zone('CET').alias('cet_datetime')
]).with_columns([
    pl.col('cet_datetime').dt.year().alias('year'),
    pl.col('cet_datetime').dt.month().alias('month'),
    pl.col('cet_datetime').dt.day().alias('day')
]).filter(
    ((pl.col('month') == 4) & (pl.col('day') >= 21)) |
    ((pl.col('month') == 5) & (pl.col('day') <= 18))
).select(['game_id', 'year']).collect()

# Total count per year (for the denominator)
yearly_totals = spring_games_metadata.group_by('year').len(name='total_games')

print(f"   -> Found {spring_games_metadata.height:,} games across the observed years.")

# Prepare a lazy version for joining with moves
valid_ids_lf = spring_games_metadata.lazy()

In [ ]:
# ==========================================
# PHASE 2: Identify Queen's Gambit Games
# ==========================================
print("⏳ Phase 2: Scanning moves for Queen's Gambit opening (1. d4 d5 2. c4)...")

qg_matches_lf = pl.scan_parquet(moves_parquet_path).filter(
    pl.col('move_no') <= 2
).join(
    valid_ids_lf, on='game_id', how='inner'
).group_by(
    'game_id'
).agg(
    pl.col('move').head(3).alias('moves_list')
).filter(
    (pl.col('moves_list').list.len() >= 3) &
    (pl.col('moves_list').list.get(0) == 'd4') &
    (pl.col('moves_list').list.get(1) == 'd5') &
    (pl.col('moves_list').list.get(2) == 'c4')
).select('game_id')

# Get the IDs of the Queen's Gambit games
qg_games_ids = qg_matches_lf.collect(streaming=True)

# Re-attach the year to the Queen's Gambit games to count them
qg_yearly_counts = qg_games_ids.join(
    spring_games_metadata, on='game_id', how='inner'
).group_by('year').len(name='qg_count')

In [ ]:
# ==========================================
# PHASE 3: Calculate and Print Final Statistics
# ==========================================
# Join the totals and the QG counts for the final report
final_report = yearly_totals.join(qg_yearly_counts, on='year', how='left').fill_null(0).sort('year')

# Calculate the percentage
final_report = final_report.with_columns(
    (pl.col('qg_count') / pl.col('total_games') * 100).alias('percentage')
)

print("-" * 60)
print("📊 RESULT: Queen's Gambit Ratio (April 21 - May 18)")
print("-" * 60)

for row in final_report.iter_rows(named=True):
    year = row['year']
    qg_c = row['qg_count']
    total = row['total_games']
    perc = row['percentage']
    print(f"📅 Year: {year}")
    print(f"   - Total Standard games: {total:,}")
    print(f"   - Queen's Gambit games: {qg_c:,}")
    print(f"   - Ratio: {perc:.4f}%")
    print("-" * 30)

print(f"🏁 Total Pipeline finished in {time.time() - start_time_total:.2f} seconds.")

### Q3

In [12]:
import polars as pl
import time

# File paths
games_parquet_path = '/content/games_full.parquet'
moves_parquet_path = '/content/moves_full.parquet'

In [ ]:
print("🚀 Starting Corrected Global Analysis (10+0 games)...")
start_time_total = time.time()

# ==========================================
# PHASE 1: Identify all 10-minute games
# ==========================================
target_games_lf = pl.scan_parquet(games_parquet_path).filter(
    pl.col('timecontrol') == '600+0'
).select(['game_id'])

In [ ]:
# ==========================================
# PHASE 2: Stream Moves and Categorize
# ==========================================
print("⏳ Scanning moves (peeking into round 4) to verify game lengths...")

analysis_results_df = pl.scan_parquet(moves_parquet_path).filter(
    pl.col('move_no') <= 4
).join(
    target_games_lf, on='game_id', how='inner'
).group_by('game_id').agg([

    # Check for King moves in the first 3 rounds
    pl.col('move').filter(
        (pl.col('move_no') <= 3) &
        (pl.col('color') == 'white') &
        (pl.col('move').str.starts_with('K'))
    ).count().alias('king_move_count'),

    # FIX: Use starts_with('O-O') to capture 'O-O', 'O-O-O', 'O-O+', and 'O-O#'
    pl.col('move').filter(
        (pl.col('move_no') <= 3) &
        (pl.col('color') == 'white') &
        (pl.col('move').str.starts_with('O-O'))
    ).count().alias('castle_count'),

    # Count total half-moves in the 4-round window
    pl.len().alias('total_plys_in_window')

]).with_columns([
    # Priority logic
    pl.when(pl.col('king_move_count') > 0)
    .then(pl.lit("Lost Rights (King Move)"))

    .when(pl.col('castle_count') > 0)
    .then(pl.lit("Castled Early"))

    # If total plys in a 4-round window is strictly less than 7,
    # it means round 4 never started. The game ended in <= 3 rounds.
    .when(pl.col('total_plys_in_window') < 7)
    .then(pl.lit("Early End (<= 3 moves)"))

    .otherwise(pl.lit("Normal Development"))
    .alias('category')
]).select('category').collect(streaming=True)

In [ ]:
# ==========================================
# PHASE 3: Generate Final Statistics
# ==========================================
final_stats = analysis_results_df.group_by('category').len(name='count').sort('count', descending=True)

# Calculate percentages
total_games = analysis_results_df.height
final_stats = final_stats.with_columns(
    (pl.col('count') / total_games * 100).alias('percentage')
)

print("-" * 60)
print("📊 CORRECTED STATISTICS (All 10+0 Games):")
print("-" * 60)
for row in final_stats.iter_rows(named=True):
    print(f"🔹 {row['category']}: {row['count']:,} games ({row['percentage']:.4f}%)")

print("-" * 60)
print(f"✅ Total 10+0 games analyzed: {total_games:,}")
print(f"🏁 Total Pipeline finished in {time.time() - start_time_total:.2f} seconds.")

### Q23

In [19]:
import polars as pl
import time

# File paths
games_parquet_path = '/content/games_full.parquet'
moves_parquet_path = '/content/moves_full.parquet'

In [ ]:
print("🚀 Starting search for players with the most castling checkmates...")
start_time_total = time.time()

# ==========================================
# PHASE 1: Filter for castling checkmates
# ==========================================
# We apply the filter directly to the full dataset using Predicate Pushdown.
# This is extremely fast as it only scans the 'move' column for specific strings.
mate_moves_lf = pl.scan_parquet(moves_parquet_path).filter(
    pl.col('move').is_in(['O-O#', 'O-O-O#'])
).select(['game_id', 'color'])

In [21]:
# ==========================================
# PHASE 2: Identify players and aggregate
# ==========================================
# Using the correct column names ('white' and 'black') based on the Parquet schema
games_metadata_lf = pl.scan_parquet(games_parquet_path).select(['game_id', 'white', 'black'])

# Join the filtered checkmates with the games metadata to get player names
top_players_df = mate_moves_lf.join(
    games_metadata_lf, on='game_id', how='inner'
).with_columns(
    # If the color of the mating move is 'white', select the 'white' player, else the 'black' player
    pl.when(pl.col('color') == 'white')
    .then(pl.col('white'))
    .otherwise(pl.col('black'))
    .alias('mate_giver')
).group_by('mate_giver').len(name='mate_count') \
 .sort(['mate_count', 'mate_giver'], descending=[True, False]) \
 .head(10).collect(engine="streaming") # Updated to the new Polars engine parameter syntax

In [ ]:
# ==========================================
# PHASE 3: Print Results
# ==========================================
print("\n" + "="*60)
print("🏆 TOP 10 PLAYERS WITH THE MOST CASTLING CHECKMATES")
print("="*60)

if top_players_df.height == 0:
    print("No castling checkmates were found in the dataset.")
else:
    for index, row in enumerate(top_players_df.iter_rows(named=True), 1):
        print(f"{index:2}. {row['mate_giver']:<20} | {row['mate_count']} castling checkmates")

print("\n" + "="*60)
print(f"🏁 Execution time: {time.time() - start_time_total:.2f} seconds")